# 蚊子音频检测模型 - 训练与测试

本 Notebook 用于蚊子音频检测模型的训练和测试，使用 HumbugDB 数据集。

## 1. 环境准备

In [ ]:
# 检查GPU是否可用
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. 克隆代码仓库

In [ ]:
# 克隆 GitHub 仓库
!git clone https://github.com/Supervisor393/mosquito_detect.git
%cd mosquito_detect

## 3. 安装依赖

In [ ]:
# 安装依赖包
!pip install -r requirements.txt -q

# 安装 PyTorch GPU 版本（如果尚未安装）
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

## 4. 下载数据集

从 Zenodo 下载 HumbugDB 数据集（4个压缩包）

In [ ]:
# 创建数据目录
!mkdir -p data/audio
!mkdir -p data/metadata

# 下载数据集压缩包
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_1.zip?download=1 -O data/audio/humbugdb_neurips_2021_1.zip
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_2.zip?download=1 -O data/audio/humbugdb_neurips_2021_2.zip
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_3.zip?download=1 -O data/audio/humbugdb_neurips_2021_3.zip
!wget https://zenodo.org/record/4904800/files/humbugdb_neurips_2021_4.zip?download=1 -O data/audio/humbugdb_neurips_2021_4.zip

# 下载元数据
!wget https://zenodo.org/record/4904800/files/neurips_2021_zenodo_0_0_1.csv?download=1 -O data/metadata/neurips_2021_zenodo_0_0_1.csv

## 5. 解压数据集

In [ ]:
# 解压所有压缩包到 audio 目录
!unzip -q data/audio/humbugdb_neurips_2021_1.zip -d data/audio/
!unzip -q data/audio/humbugdb_neurips_2021_2.zip -d data/audio/
!unzip -q data/audio/humbugdb_neurips_2021_3.zip -d data/audio/
!unzip -q data/audio/humbugdb_neurips_2021_4.zip -d data/audio/

# 查看解压后的文件数量
!ls data/audio/*.wav | wc -l

## 6. 数据预处理

运行 prepare_data.py 脚本，将数据划分为训练集、验证集和测试集

In [ ]:
# 运行数据预处理脚本
!python prepare_data.py --source_dir data/audio --metadata_path data/metadata/neurips_2021_zenodo_0_0_1.csv --output_dir data

# 查看数据集划分结果
print("\n训练集:")
!ls data/train/mosquito/ | wc -l
!ls data/train/no_mosquito/ | wc -l

print("\n验证集:")
!ls data/val/mosquito/ | wc -l
!ls data/val/no_mosquito/ | wc -l

print("\n测试集:")
!ls data/test/mosquito/ | wc -l
!ls data/test/no_mosquito/ | wc -l

## 7. 训练模型

In [ ]:
# 训练模型
!python train.py --model efficient --batch_size 64 --epochs 50 --lr 1e-4 --device cuda

## 8. 测试模型

In [ ]:
# 测试模型
!python test.py --model_path models/mosquito_detector_efficient_best.pth --model_type efficient --device cuda

## 9. 模型推理示例

In [ ]:
# 导入必要的模块
import torch
import os
from model import get_model
from inference import AudioProcessor, predict_audio
import config

# 加载模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_model('efficient', num_classes=config.NUM_CLASSES).to(device)
model.load_state_dict(torch.load('models/mosquito_detector_efficient_best.pth', map_location=device))
model.eval()

# 创建音频处理器
audio_processor = AudioProcessor()

# 测试一些音频文件
test_files = [
    'data/test/mosquito/' + os.listdir('data/test/mosquito/')[0] if os.listdir('data/test/mosquito/') else None,
    'data/test/no_mosquito/' + os.listdir('data/test/no_mosquito/')[0] if os.listdir('data/test/no_mosquito/') else None
]

for file_path in test_files:
    if file_path and os.path.exists(file_path):
        prediction, confidence, num_segments = predict_audio(model, audio_processor, file_path, device)
        label = '蚊子' if prediction == 1 else '无蚊子'
        print(f"\n文件: {os.path.basename(file_path)}")
        print(f"预测结果: {label}")
        print(f"置信度: {confidence:.4f}")
        print(f"处理片段数: {num_segments}")

## 10. 模型结构可视化

In [ ]:
# 可视化模型结构
from model import EfficientMosquitoDetector
import torchsummary

model = EfficientMosquitoDetector(num_classes=2)
torchsummary.summary(model, input_size=(1, 64, 100))